# A2 — Image Classification: Custom CNN vs ResNet-18 vs EfficientNet-B0


## 1. Imports & Configuration

In [7]:
import os, csv, time, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision import models
from torchvision.transforms import v2

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
)

import matplotlib
matplotlib.use("Agg")   # remove for inline Jupyter display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# ── Paths (must match data_generation_Finn.ipynb) ─────────────────────────
DATASET_ROOT = "dataset"
OUTPUT_DIR   = Path("outputs"); OUTPUT_DIR.mkdir(exist_ok=True)

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 3
CLASS_NAMES = ["1 die", "2 dice", "3 dice"]   # sym_num_dice = 1, 2, 3 → class 0, 1, 2

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

# Verify dataset exists with correct sym_* schema
for split in ["train", "val", "test"]:
    p = Path(DATASET_ROOT) / split / "labels.csv"
    assert p.exists(), f"Missing: {p} — run data_generation_Finn.ipynb first."
    df = pd.read_csv(p, nrows=1)
    required = {"image", "text", "sym_num_dice", "sym_values", "sym_colours", "sym_sizes"}
    missing  = required - set(df.columns)
    assert not missing, (
        f"CSV missing columns {missing}.\n"
        "Run data_generation_Finn.ipynb (not the old generator) to get sym_* fields."
    )
print("Dataset schema valid. Ready.")


Device : cpu
PyTorch: 2.10.0
Dataset schema valid. Ready.


## 2. DiceDataset


In [8]:
# ── Symbol index maps (must match data_generation_Finn.ipynb) ─────────────
VALID_COLOURS = ["white", "red", "blue", "green", "yellow", "purple", "peach"]
VALID_SIZES   = ["small", "medium", "large"]
COLOUR_TO_IDX = {c: i for i, c in enumerate(VALID_COLOURS)}
SIZE_TO_IDX   = {s: i for i, s in enumerate(VALID_SIZES)}


class DiceDataset(Dataset):
    """
    PyTorch Dataset for the synthetic dice dataset produced by data_generation_Finn.ipynb.

    Each __getitem__ returns:
        image   — float32 tensor [3, H, W], transformed
        labels  — dict with keys:
                    text          : str
                    sym_num_dice  : scalar long tensor  (1–3)
                    sym_values    : long tensor [3], padded with 0
                    sym_colours   : long tensor [3], padded with -1
                    sym_sizes     : long tensor [3], padded with -1
    """

    VALID_SPLITS = ("train", "val", "test")

    def __init__(self, root_dir: str, split: str = "train", transform=None):
        if split not in self.VALID_SPLITS:
            raise ValueError(f"split must be one of {self.VALID_SPLITS}, got '{split}'.")

        self.image_dir = os.path.join(root_dir, split, "images")
        self.csv_path  = os.path.join(root_dir, split, "labels.csv")

        assert os.path.isdir(self.image_dir), f"Image dir not found: {self.image_dir}"
        assert os.path.isfile(self.csv_path), f"Labels CSV not found: {self.csv_path}"

        self.samples = []
        with open(self.csv_path, "r", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                self.samples.append(row)
        assert len(self.samples) > 0, "No samples found in CSV."

        # Default transform: bare tensor conversion (overridden below)
        self.transform = transform or v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
        ])

        print(f"DiceDataset ({split}): {len(self.samples)} samples loaded.")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        row      = self.samples[idx]
        img_path = os.path.join(self.image_dir, row["image"])
        image    = Image.open(img_path).convert("RGB")
        image    = self.transform(image)

        # Parse and validate symbol fields
        num_dice = int(row["sym_num_dice"])
        values   = list(map(int, row["sym_values"].split()))
        colours  = row["sym_colours"].split()
        sizes    = row["sym_sizes"].split()

        # Pad to length 3 (max dice count)
        padded_values  = values  + [0]  * (3 - len(values))
        padded_colours = [COLOUR_TO_IDX[c] for c in colours] + [-1] * (3 - len(colours))
        padded_sizes   = [SIZE_TO_IDX[s]   for s in sizes]   + [-1] * (3 - len(sizes))

        label = {
            "text":         row["text"],
            "sym_num_dice": torch.tensor(num_dice,       dtype=torch.long),
            "sym_values":   torch.tensor(padded_values,  dtype=torch.long),
            "sym_colours":  torch.tensor(padded_colours, dtype=torch.long),
            "sym_sizes":    torch.tensor(padded_sizes,   dtype=torch.long),
        }
        return image, label


def dice_collate_fn(batch):
    """Custom collate to handle the variable-length 'text' string field."""
    images = torch.stack([item[0] for item in batch])
    labels = {
        "text":         [item[1]["text"]         for item in batch],
        "sym_num_dice": torch.stack([item[1]["sym_num_dice"] for item in batch]),
        "sym_values":   torch.stack([item[1]["sym_values"]   for item in batch]),
        "sym_colours":  torch.stack([item[1]["sym_colours"]  for item in batch]),
        "sym_sizes":    torch.stack([item[1]["sym_sizes"]     for item in batch]),
    }
    return images, labels


## 3. Data Loaders

In [9]:
IMG_SIZE      = 224
BATCH_SIZE    = 32   # larger dataset (4181 train) supports bigger batches
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = DiceDataset(DATASET_ROOT, "train", train_transform)
val_ds   = DiceDataset(DATASET_ROOT, "val",   eval_transform)
test_ds  = DiceDataset(DATASET_ROOT, "test",  eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=dice_collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=dice_collate_fn, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=dice_collate_fn, num_workers=0)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

# Verify a batch has the right shape and sym_* keys
imgs, labels = next(iter(train_loader))
print(f"Image batch shape : {imgs.shape}")              # [32, 3, 224, 224]
print(f"sym_num_dice shape: {labels['sym_num_dice'].shape}")
print(f"sym_values shape  : {labels['sym_values'].shape}")
print(f"sym_colours shape : {labels['sym_colours'].shape}")
print(f"sym_sizes shape   : {labels['sym_sizes'].shape}")
print(f"Example text      : {labels['text'][0]}")

# Class balance
counts = Counter(int(row["sym_num_dice"]) for row in train_ds.samples)
print(f"Train class dist  : { {CLASS_NAMES[k-1]: v for k, v in sorted(counts.items())} }")


DiceDataset (train): 4181 samples loaded.
DiceDataset (val): 899 samples loaded.
DiceDataset (test): 898 samples loaded.
Train batches: 131 | Val: 29 | Test: 29
Image batch shape : torch.Size([32, 3, 224, 224])
sym_num_dice shape: torch.Size([32])
sym_values shape  : torch.Size([32, 3])
sym_colours shape : torch.Size([32, 3])
sym_sizes shape   : torch.Size([32, 3])
Example text      : The scene contains a large red die showing six, a medium red die showing six and a medium red die showing four.
Train class dist  : {'1 die': 1384, '2 dice': 1413, '3 dice': 1384}


## 4. Model Definitions

### a. CustomCNN — 4 convolutional blocks from scratch
Conv2d → BatchNorm2d → ReLU → MaxPool2d, repeated 4 times.
AdaptiveAvgPool to a fixed spatial size, then a two-layer FC head with dropout.
~0.3M parameters. No pretrained weights.

### b. ResNet-18 — fine-tuned from ImageNet
Standard ResNet-18 architecture with skip connections. The final fully-connected
layer is replaced with a 3-class head. All layers are unfrozen and fine-tuned
end-to-end. Uses a lower learning rate than the CNN to preserve ImageNet features.

### c. EfficientNet-B0 — fine-tuned from ImageNet
Compound-scaled architecture (balanced depth, width, resolution). Classifier
head replaced with a 3-class linear layer. All layers fine-tuned.


In [10]:
# ── 4a. CustomCNN ─────────────────────────────────────────────────────────
class CustomCNN(nn.Module):
    """
    4-block CNN built entirely from scratch.
    Architecture: [Conv-BN-ReLU-MaxPool] x4 → AdaptiveAvgPool → FC head.
    ~0.3M parameters.
    """
    def __init__(self, num_classes: int = 3):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 3 → 32  |  224 → 112
            nn.Conv2d(3,   32,  kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            # Block 2: 32 → 64  |  112 → 56
            nn.Conv2d(32,  64,  kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            # Block 3: 64 → 128  |  56 → 28
            nn.Conv2d(64,  128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            # Block 4: 128 → 256  |  AdaptiveAvgPool → 4×4
            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


# ── 4b. ResNet-18 ──────────────────────────────────────────────────────────
def build_resnet18(pretrained: bool = True) -> nn.Module:
    """
    ResNet-18. When pretrained=True, loads ImageNet weights (requires network).
    Falls back to random init with a clear warning if download fails.
    Final FC replaced with a 3-class head. All layers fine-tuned.
    """
    try:
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        model   = models.resnet18(weights=weights)
        if pretrained:
            print("ResNet-18: ImageNet weights loaded successfully.")
    except Exception as e:
        print(f"ResNet-18 WARNING: could not load pretrained weights ({e}).\n"
              "         Falling back to random init. Results will differ from "
              "full fine-tuning experiment.")
        model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model


# ── 4c. EfficientNet-B0 ────────────────────────────────────────────────────
def build_efficientnet_b0(pretrained: bool = True) -> nn.Module:
    """
    EfficientNet-B0. Pretrained weights loaded when pretrained=True.
    Classifier head replaced with a 3-class linear layer. All layers fine-tuned.
    """
    try:
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model   = models.efficientnet_b0(weights=weights)
        if pretrained:
            print("EfficientNet-B0: ImageNet weights loaded successfully.")
    except Exception as e:
        print(f"EfficientNet-B0 WARNING: could not load pretrained weights ({e}).\n"
              "              Falling back to random init.")
        model = models.efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    return model


# ── Parameter counts ───────────────────────────────────────────────────────
def count_params(m: nn.Module) -> str:
    return f"{sum(p.numel() for p in m.parameters())/1e6:.2f}M"

_c = CustomCNN()
_r = models.resnet18(weights=None); _r.fc = nn.Linear(512, 3)
_e = models.efficientnet_b0(weights=None); _e.classifier[1] = nn.Linear(1280, 3)
print(f"CustomCNN      : {count_params(_c)}")
print(f"ResNet-18      : {count_params(_r)}")
print(f"EfficientNet-B0: {count_params(_e)}")
del _c, _r, _e


CustomCNN      : 1.44M
ResNet-18      : 11.18M
EfficientNet-B0: 4.01M


## 5. Training Infrastructure


In [11]:
def get_predictions(model: nn.Module, loader: DataLoader) -> tuple:
    """
    Run inference over a full DataLoader.
    Returns (predictions array, true labels array, mean loss).
    Labels are extracted from label_dict['sym_num_dice'] and shifted to 0-index.
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()
    all_preds, all_labels = [], []
    total_loss, n = 0.0, 0

    with torch.no_grad():
        for imgs, label_dict in loader:
            imgs   = imgs.to(DEVICE)
            # sym_num_dice is 1-indexed (1,2,3) — shift to 0-indexed (0,1,2)
            labels = (label_dict["sym_num_dice"] - 1).to(DEVICE)
            logits = model(imgs)
            total_loss += criterion(logits, labels).item() * len(imgs)
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            n += len(imgs)

    return np.array(all_preds), np.array(all_labels), total_loss / n


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Return accuracy, macro precision/recall/F1, and confusion matrix."""
    return {
        "accuracy":  round(accuracy_score(y_true, y_pred), 4),
        "precision": round(precision_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "recall":    round(recall_score(y_true,    y_pred, average="macro", zero_division=0), 4),
        "f1_macro":  round(f1_score(y_true,        y_pred, average="macro", zero_division=0), 4),
        "cm":        confusion_matrix(y_true, y_pred, labels=[0, 1, 2]),
    }


def train_model(
    model:        nn.Module,
    name:         str,
    epochs:       int   = 30,
    lr:           float = 1e-3,
    weight_decay: float = 1e-4,
) -> dict:
    """
    Full training loop with cosine LR scheduling and best-checkpoint saving.

    The training step pulls sym_num_dice from label_dict and shifts to 0-index.
    Returns a results dict with epoch log, val/test metrics, predictions, and timing.
    """
    model     = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_acc = 0.0
    best_state   = None
    epoch_log    = []
    t0           = time.time()

    print(f"\n{'='*60}")
    print(f"  Training : {name}")
    print(f"  Epochs: {epochs}  |  LR: {lr}  |  Device: {DEVICE}")
    print(f"{'='*60}")
    print(f"{'Epoch':>6}  {'Train Acc':>10}  {'Val Acc':>9}  {'Val Loss':>9}")

    for ep in range(1, epochs + 1):
        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        correct, n = 0, 0
        for imgs, label_dict in train_loader:
            imgs   = imgs.to(DEVICE)
            labels = (label_dict["sym_num_dice"] - 1).to(DEVICE)   # 0-indexed
            optimizer.zero_grad()
            logits = model(imgs)
            criterion(logits, labels).backward()
            optimizer.step()
            correct += (logits.argmax(1) == labels).sum().item()
            n       += len(imgs)
        train_acc = correct / n

        # ── Validate ───────────────────────────────────────────────────────
        val_preds, val_labels, val_loss = get_predictions(model, val_loader)
        val_acc = accuracy_score(val_labels, val_preds)
        scheduler.step()

        epoch_log.append({
            "epoch":     ep,
            "train_acc": round(train_acc, 4),
            "val_acc":   round(val_acc,   4),
            "val_loss":  round(val_loss,  4),
        })

        if val_acc >= best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if ep % 5 == 0 or ep == 1:
            print(f"{ep:>6}  {train_acc:>10.4f}  {val_acc:>9.4f}  {val_loss:>9.4f}")

    # ── Evaluate best checkpoint on test set ───────────────────────────────
    model.load_state_dict(best_state)
    test_preds, test_labels, _ = get_predictions(model, test_loader)
    val_preds2, val_labels2, _ = get_predictions(model, val_loader)
    elapsed = time.time() - t0

    test_metrics = compute_metrics(test_labels, test_preds)
    val_metrics  = compute_metrics(val_labels2, val_preds2)

    print(f"\n  Best val acc : {best_val_acc:.4f}")
    print(f"  Test acc     : {test_metrics['accuracy']:.4f}")
    print(f"  Test F1 (mac): {test_metrics['f1_macro']:.4f}")
    print(f"  Time         : {elapsed:.1f}s")
    print("\n  Per-class classification report (test set):")
    print(classification_report(test_labels, test_preds,
                                target_names=CLASS_NAMES, digits=3))
    return {
        "log":          epoch_log,
        "val_metrics":  val_metrics,
        "test_metrics": test_metrics,
        "test_preds":   test_preds,
        "test_labels":  test_labels,
        "best_val_acc": best_val_acc,
        "train_time_s": elapsed,
    }


## 6. Run All Axis 2 Experiments

In [ ]:
torch.manual_seed(SEED)

experiments = [
    # (name,             model,                       epochs, lr,   weight_decay)
    ("CustomCNN",        CustomCNN(NUM_CLASSES),       30,     1e-3, 1e-4),
    ("ResNet-18",        build_resnet18(True),         30,     5e-4, 1e-4),
    ("EfficientNet-B0",  build_efficientnet_b0(True),  30,     5e-4, 1e-4),
]

all_results = {}

for name, model, epochs, lr, wd in experiments:
    torch.manual_seed(SEED)
    result = train_model(model, name, epochs=epochs, lr=lr, weight_decay=wd)
    all_results[name] = result

    # Save per-epoch log to CSV
    log_path = OUTPUT_DIR / f"axis2_log_{name.replace(' ','_').replace('-','')}.csv"
    pd.DataFrame(result["log"]).to_csv(log_path, index=False)
    print(f"  Epoch log saved → {log_path}\n")


ResNet-18: ImageNet weights loaded successfully.
EfficientNet-B0: ImageNet weights loaded successfully.

  Training : CustomCNN
  Epochs: 30  |  LR: 0.001  |  Device: cpu
 Epoch   Train Acc    Val Acc   Val Loss
     1      0.8113     0.4138     2.6210
     5      0.9859     0.9978     0.0106
    10      0.9912     1.0000     0.0009
    15      0.9923     1.0000     0.0024
    20      0.9969     0.9989     0.0016
    25      0.9974     1.0000     0.0006
    30      0.9981     1.0000     0.0001

  Best val acc : 1.0000
  Test acc     : 1.0000
  Test F1 (mac): 1.0000
  Time         : 8297.7s

  Per-class classification report (test set):
              precision    recall  f1-score   support

       1 die      1.000     1.000     1.000       302
      2 dice      1.000     1.000     1.000       308
      3 dice      1.000     1.000     1.000       288

    accuracy                          1.000       898
   macro avg      1.000     1.000     1.000       898
weighted avg      1.000     1.

## 7. Axis 2 Results Table

In [ ]:
rows = []
for name, result in all_results.items():
    tm = result["test_metrics"]
    rows.append({
        "Model":           name,
        "Pretrained":      "No (scratch)" if name == "CustomCNN" else "Yes (ImageNet)",
        "Val Acc":         round(result["best_val_acc"], 4),
        "Test Acc":        tm["accuracy"],
        "Precision (mac)": tm["precision"],
        "Recall (mac)":    tm["recall"],
        "F1 (macro)":      tm["f1_macro"],
        "Train Time (s)":  round(result["train_time_s"], 1),
    })

results_df = pd.DataFrame(rows)

print("=" * 80)
print("  AXIS 2 — Results Table")
print("=" * 80)
print(results_df.to_string(index=False))

results_path = OUTPUT_DIR / "axis2_results.csv"
results_df.to_csv(results_path, index=False)
print(f"\nSaved → {results_path}")


## 8. Confusion Matrices — Test Set


In [ ]:
BG      = "#0f172a"
TEXT    = "#e2e8f0"
MUTED   = "#64748b"
PALETTE = {
    "CustomCNN":       "#3b82f6",
    "ResNet-18":       "#22c55e",
    "EfficientNet-B0": "#f59e0b",
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=BG)
fig.suptitle("Confusion Matrices — Test Set (Best Val Checkpoint)",
             color="white", fontsize=13, fontweight="bold", y=1.02)

for ax, (name, result) in zip(axes, all_results.items()):
    cm_raw  = result["test_metrics"]["cm"]
    cm_norm = cm_raw.astype(float) / cm_raw.sum(axis=1, keepdims=True).clip(min=1)
    c       = PALETTE[name]
    tacc    = result["test_metrics"]["accuracy"]
    tf1     = result["test_metrics"]["f1_macro"]

    sns.heatmap(
        cm_norm, ax=ax,
        cmap=sns.light_palette(c, as_cmap=True),
        annot=cm_raw, fmt="d",
        linewidths=0.5, linecolor="#334155",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        cbar=False,
        annot_kws={"size": 13, "weight": "bold", "color": "white"},
    )
    ax.set_facecolor(BG)
    ax.set_title(f"{name}\nAcc {tacc:.3f}  |  F1 {tf1:.3f}",
                 color=c, fontsize=11, fontweight="bold", pad=10)
    ax.set_xlabel("Predicted", color=MUTED, fontsize=9)
    ax.set_ylabel("True",      color=MUTED, fontsize=9)
    ax.tick_params(colors=MUTED, labelsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor("#334155")

plt.tight_layout()
cm_path = OUTPUT_DIR / "axis2_confusion_matrices.png"
fig.savefig(cm_path, dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Saved → {cm_path}")


## 9. Learning Curves & Comparative Plots

In [ ]:
fig = plt.figure(figsize=(17, 10), facecolor=BG)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.32,
                         left=0.07, right=0.97, top=0.88, bottom=0.10)

def style_ax(ax):
    ax.set_facecolor(BG)
    for sp in ax.spines.values(): sp.set_edgecolor("#334155")
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.yaxis.grid(True, color="#1e293b", linestyle="--", lw=0.6)
    ax.set_axisbelow(True)
    ax.xaxis.label.set_color(MUTED)
    ax.yaxis.label.set_color(MUTED)

# Row 0: individual learning curves
for col, (name, result) in enumerate(all_results.items()):
    ax  = fig.add_subplot(gs[0, col]); style_ax(ax)
    log = result["log"]
    ep  = [r["epoch"]     for r in log]
    tr  = [r["train_acc"] for r in log]
    va  = [r["val_acc"]   for r in log]
    c   = PALETTE[name]
    ax.plot(ep, tr, color=c, lw=2,                     label="Train acc")
    ax.plot(ep, va, color=c, lw=2, ls="--", alpha=0.7, label="Val acc")
    ax.axhline(result["test_metrics"]["accuracy"],
               color="#ef4444", lw=1.2, ls=":",
               label=f"Test {result['test_metrics']['accuracy']:.3f}")
    ax.set_ylim(0.2, 1.10)
    ax.set_xlabel("Epoch", fontsize=9); ax.set_ylabel("Accuracy", fontsize=9)
    ax.set_title(name, color=c, fontsize=11, fontweight="bold", pad=7)
    ax.legend(fontsize=8, facecolor="#1e293b", edgecolor="#334155", labelcolor="#cbd5e1")

# Row 1 left: grouped metric bar chart (Acc / Precision / Recall / F1)
ax_m = fig.add_subplot(gs[1, 0]); style_ax(ax_m)
metric_keys  = ["accuracy", "precision", "recall", "f1_macro"]
metric_labels= ["Accuracy", "Precision", "Recall", "F1 (macro)"]
x = np.arange(len(metric_keys)); w = 0.22
for i, (name, result) in enumerate(all_results.items()):
    tm   = result["test_metrics"]
    vals = [tm[k] for k in metric_keys]
    bars = ax_m.bar(x + i*w, vals, w, color=PALETTE[name], label=name, zorder=3, alpha=0.9)
    for bar, v in zip(bars, vals):
        ax_m.text(bar.get_x()+bar.get_width()/2, v+0.008, f"{v:.2f}",
                  ha="center", va="bottom", color="white", fontsize=6.5, fontweight="bold")
ax_m.set_xticks(x + w); ax_m.set_xticklabels(metric_labels, fontsize=8, color=MUTED)
ax_m.set_ylim(0, 1.22); ax_m.set_ylabel("Score", fontsize=9)
ax_m.set_title("All Metrics — Test Set", color=TEXT, fontsize=11, fontweight="bold", pad=7)
ax_m.legend(fontsize=7, facecolor="#1e293b", edgecolor="#334155", labelcolor="#cbd5e1")

# Row 1 mid: val accuracy convergence overlay
ax_c = fig.add_subplot(gs[1, 1]); style_ax(ax_c)
for name, result in all_results.items():
    log = result["log"]
    ax_c.plot([r["epoch"] for r in log], [r["val_acc"] for r in log],
              color=PALETTE[name], lw=2, label=name)
ax_c.set_ylim(0.2, 1.10)
ax_c.set_xlabel("Epoch", fontsize=9); ax_c.set_ylabel("Val Accuracy", fontsize=9)
ax_c.set_title("Convergence Comparison
(Val Accuracy)", color=TEXT, fontsize=11, fontweight="bold", pad=7)
ax_c.legend(fontsize=8, facecolor="#1e293b", edgecolor="#334155", labelcolor="#cbd5e1")

# Row 1 right: params vs test F1 scatter
ax_s = fig.add_subplot(gs[1, 2]); style_ax(ax_s)
ax_s.xaxis.grid(True, color="#1e293b", linestyle="--", lw=0.6)
param_M = {"CustomCNN": 0.3, "ResNet-18": 11.2, "EfficientNet-B0": 5.3}
for name, result in all_results.items():
    f1 = result["test_metrics"]["f1_macro"]
    ax_s.scatter(param_M[name], f1, color=PALETTE[name], s=140, zorder=5)
    ax_s.annotate(name, (param_M[name], f1),
                  textcoords="offset points", xytext=(6, 4),
                  color="#cbd5e1", fontsize=8)
ax_s.set_xlabel("Parameters (M)", fontsize=9); ax_s.set_ylabel("Test F1 (macro)", fontsize=9)
ax_s.set_title("Params vs. F1 (macro)", color=TEXT, fontsize=11, fontweight="bold", pad=7)

fig.suptitle(
    "Axis 2 — CustomCNN vs ResNet-18 vs EfficientNet-B0  |  Synthetic Dice Dataset (sym_num_dice)",
    color="white", fontsize=12, fontweight="bold", y=0.96
)
curves_path = OUTPUT_DIR / "axis2_learning_curves.png"
fig.savefig(curves_path, dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Saved → {curves_path}")


## 10. Per-Class Metrics Breakdown — Test Set

In [ ]:
print("=" * 65)
print("  PER-CLASS METRICS — TEST SET (best val checkpoint)")
print("=" * 65)

for name, result in all_results.items():
    print(f"\n{'─'*48}")
    print(f"  {name}")
    print(f"{'─'*48}")
    print(classification_report(
        result["test_labels"],
        result["test_preds"],
        target_names=CLASS_NAMES,
        digits=3,
    ))
